Notebook per la creazione e l'analisi del dataset scaricato da PDB: PL- v.2020R1

1. estrazione informazioni da INDEX_general_PL.2020R1.ist
2. creazione csv
3. salvataggio file corretti pdb per proteina e sdf per ligand
4. analisi e visualizzazione
5. 




In [1]:
import os
import re #regex library
import csv
import pandas as pd
import numpy as np

#ignore warinings
import warnings
warnings.filterwarnings("ignore")

## **Analisi e Scraping Dataset**


### **Estrazione informazioni INDEX**

In [ ]:
# read and extract info from INDEX_general_PL
datapath = "data\\PDB"
filename = "INDEX_general_PL.2020R1.lst"
filepath = os.path.join(datapath, filename)

if not os.path.exists(filepath):
    raise FileNotFoundError(f"{filename} file not found path[{filepath}]")

#read file
with open(filepath, "r", encoding="utf-8") as f:
    lines = [
        ln.strip()
        for ln in f
        if ln.strip() and not ln.lstrip().startswith("#")
    ]
    
#regex [PDB_code, resolution, release_year, binding_data[type,compare,unit], reference, ligand_name, comments]
regex= re.compile(r"(\w+)\s+"+                              #pdb code
                  r"([\d.]+|NMR)\s+"+                       #resolution
                  r"(\d{4})\s+"+                            #release year
                  r"(\w+)([=<~]{0,2})([\d\.]+)(\w+)\s+"     #binding data
                  r"//\s[\w\.]+\s+"                         #file path reference (ignored)
                  r"\(([^)]+?)\)"                           #ligand name
                  r"\s*(.*)"                                #comments
                  )

parsed_index = []
skipped = 0
for ln in lines:
    match = regex.match(ln)
    if match:
        (pdb, res, year, btype, comp, val, unit, ligand, comment) = match.groups()
        
        parsed_index.append({
            "PDB_code": pdb,
            "resolution": res,
            "release_year": year,
            "binding_type": btype,
            "compare": comp,
            "value": val,
            "unit": unit,
            "ligand_name": ligand,
            "comments": comment.strip()
        })
    else:
        skipped += 1
        #print("SKIPPED:", ln)
print(f"Parsed: {len(parsed_index)} | Skipped: {skipped}")

#salvo csv
csv_filename = "parsed_index.csv"
csv_filepath = os.path.join(datapath, csv_filename)

with open(csv_filepath, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=parsed_index[0].keys())
    writer.writeheader()
    writer.writerows(parsed_index)
       


Parsed: 19037 | Skipped: 0


### **Pulizia dataframe**

In [24]:
#dataframe info
datapath = "data\\PDB"
csv_filename = "parsed_index.csv"
csv_filepath = os.path.join(datapath, csv_filename)

df = pd.read_csv(csv_filepath)
initial_entries = len(df)
#df.info()
#df.describe()

#CLEANING OF DATA
#remove NMR entries
df = df[df["resolution"] != "NMR"]
#remove <, >, ~ from compare and value
df = df[df["compare"] == "="]
#convert resolution to float
df["resolution"] = df["resolution"].astype(np.float32)
#convert release_year to int
df["release_year"] = df["release_year"].astype(int)
#convert value to float
df["value"] = df["value"].astype(np.float32)

#remove entries with resolution > 2.5 Å
df = df[df["resolution"] <= 2.5]

#remove comments with "no ligand" or "no binding" ... (forse meglio togliere tutte le righe con commenti)
invalid_comments=["incomplete ligand",
                  "covalent complex",
                  "covalent bond",
                  "isomer",
                  "Different protein in assay",
                  "Different ligand in assay",
                  "Nonstandard assay",
                  ]
df = df[~df["comments"].str.contains("|".join(invalid_comments), case=False, na=False)]
remaining_comments = df["comments"].unique()
#print(remaining_comments)


#ANALYSIS OF BINDING DATA
binding_counts = df["binding_type"].value_counts()
binding_percentages = df["binding_type"].value_counts(normalize=True) * 100
for btype, count in binding_counts.items():
    percentage = binding_percentages[btype]
    print(f"{btype}\t: {count} entries ({percentage:.2f}%)")
    
#dizionario per la conversione in Molarità (M)
unit_multiplier = {
    'mM': 1e-3,
    'uM': 1e-6,
    'nM': 1e-9,
    'pM': 1e-12,
    'fM': 1e-15
}

#conversione e aggiunta colonna molarità [value_M]
df["value_M"] = df["value"] * df["unit"].map(unit_multiplier).fillna(1) 

# uniformare le 3 misure calcolando il logaritmo negativo (pAffinity)
# questo trasforma Kd, Ki e IC50 in una singola scala pKd / pKi / pIC50
df=df[df["value_M"] > 0]  #rimuovo valori non validi (zero o negativi)
df['pAffinity'] = -np.log10(df['value_M']).astype(np.float32).round(4)

# OPXIONALE: IC50 è meno affidabile come misura di affinità possiamo scartarlo e mantenere solo Kd e Ki (35% abbiamo molti meno dati)
#df = df[df['binding_type'].isin(['Kd', 'Ki'])]

print(df.head(10))

#risultati cleaning:
print("\nData Cleaning Results:")
entries_removed_percentage = (initial_entries - len(df)) / initial_entries * 100
print(f"Entries before cleaning: {initial_entries}")
print(f"Entries after cleaning: {len(df)}")
print(f"Removed: {initial_entries - len(df)} ({entries_removed_percentage:.2f}%)")


#Drop useles colums and save cleaned dataset
cleaned_csv_filename = "PL_cleaned_index.csv"
cleaned_csv_filepath = os.path.join(datapath, cleaned_csv_filename)
df.drop(columns=["binding_type","compare", "value","unit", "value_M","ligand_name", "comments", ], inplace=True)
df.to_csv(cleaned_csv_filepath, index=False)

IC50	: 4931 entries (36.62%)
Kd	: 4853 entries (36.04%)
Ki	: 3682 entries (27.34%)
   PDB_code  resolution  release_year binding_type compare    value unit  \
0      2tpi         2.1          1982           Kd       =   49.000   uM   
2      4tln         2.3          1982           Ki       =  190.000   uM   
5      1rnt         1.9          1987           Kd       =    6.500   uM   
7      4ts1         2.5          1989           Kd       =   11.600   uM   
8      4tmn         1.7          1989           Ki       =    0.068   nM   
9      2tmn         1.6          1989           Ki       =    1.300   uM   
10     1tlp         2.3          1989           Ki       =   28.000   nM   
11     1tmn         1.9          1989           Ki       =   50.000   nM   
12     5tmn         1.6          1989           Ki       =    9.100   nM   
25     1rbp         2.0          1991           Kd       =    0.190   uM   

   ligand_name comments       value_M  pAffinity  
0        2-mer      NaN  4.90

## **Data Retrival**